# FlowThought PoC

**Flow Matching for Reasoning in LLM Hidden State Space**

This notebook implements the FlowThought proof-of-concept:
1. **Phase 1** — Extract hidden states from Qwen2.5-0.5B-Instruct on GSM8K
2. **Phase 2** — Train a CFM velocity field (VelocityMLP) to map noise -> CoT hidden states
3. **Phase 3a** — Evaluate via answer probe: compare no-CoT, real CoT, random, and FlowThought
4. **Phase 3b** — Coconut-style hidden state injection: add FM-generated states to the LLM residual stream during generation and measure downstream accuracy

Uses vanilla Conditional Flow Matching (independent coupling, not OT-CFM). Runs end-to-end on a free Colab T4 GPU.

In [1]:
# Cell 1: Install dependencies (torch is pre-installed on Colab)
!pip install -q transformers==4.47.0 datasets==3.2.0 accelerate==1.2.1 matplotlib==3.9.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.4/336.4 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 28.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
# Cell 2: Imports + Config
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import json
import os
import re
import gc
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Optional, Tuple

CONFIG = {
    "model_name": "Qwen/Qwen2.5-0.5B-Instruct",
    "hidden_dim": 896,
    "n_train": 2000,             # smoke test — scale to 2000 after validation
    "n_test": 500,               # smoke test — scale to 500 after validation
    "batch_size_extract": 32,   # for short texts (problem-only)
    "batch_size_extract_long": 4,  # for long texts (problem+CoT) — T4 safe
    "max_length": 512,
    # FM training
    "fm_epochs": 300,            # smoke test — scale to 300 after validation
    "fm_batch_size": 2000,       # full dataset per epoch
    "fm_lr": 3e-4,
    "fm_hidden": 1024,
    "fm_layers": 4,
    "grad_clip": 1.0,
    "fm_checkpoint_every": 25,
    "time_embed_dim": 64,
    # ODE
    "ode_steps": 20,
    # Probe
    "probe_epochs": 100,         # smoke test — scale to 100 after validation
    "probe_lr": 1e-3,
    "probe_hidden": 256,
    # Paths
    "cache_dir": "flowthought_cache",
}

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [3]:
# Cell 3: Test framework
@dataclass
class TestResults:
    passed: int = 0
    failed: int = 0
    warned: int = 0
    details: List[str] = field(default_factory=list)

    def check(self, condition: bool, name: str):
        if condition:
            self.passed += 1
            self.details.append(f"  PASS: {name}")
        else:
            self.failed += 1
            self.details.append(f"  FAIL: {name}")

    def warn(self, condition: bool, name: str):
        """Soft check — logs WARN instead of FAIL, does not block execution."""
        if condition:
            self.passed += 1
            self.details.append(f"  PASS: {name}")
        else:
            self.warned += 1
            self.details.append(f"  WARN: {name}")

    def summary(self, section: str):
        total = self.passed + self.failed + self.warned
        status = "ALL PASSED" if self.failed == 0 and self.warned == 0 else ""
        if self.failed > 0:
            status = f"{self.failed} FAILED"
        elif self.warned > 0:
            status = f"PASSED with {self.warned} warning(s)"
        print(f"\n[{section}] {self.passed}/{total} tests passed — {status}")
        for d in self.details:
            print(d)
        assert self.failed == 0, f"{self.failed} tests failed in {section}"

# Quick self-test
t = TestResults()
t.check(True, "framework works")
t.summary("Self-test")


[Self-test] 1/1 tests passed — ALL PASSED
  PASS: framework works


In [4]:
# Cell 4: Data loading — GSM8K + answer parsing
from datasets import load_dataset

ds = load_dataset("openai/gsm8k", "main")
print(f"Train: {len(ds['train'])}, Test: {len(ds['test'])}")

def parse_answer(answer_str: str) -> Optional[float]:
    """Extract the numeric answer after #### from GSM8K format."""
    match = re.search(r"####\s*([\-\d,\.]+)", answer_str)
    if match:
        return float(match.group(1).replace(",", ""))
    return None

def get_cot_and_answer(example):
    """Split GSM8K answer field into CoT reasoning and numeric answer."""
    text = example["answer"]
    parts = text.split("####")
    cot = parts[0].strip()
    ans = parse_answer(text)
    return cot, ans

# Tests
t = TestResults()
t.check(parse_answer("blah blah\n#### 42") == 42.0, "parse simple")
t.check(parse_answer("stuff\n#### 1,234") == 1234.0, "parse comma")
t.check(parse_answer("stuff\n#### -5") == -5.0, "parse negative")
t.check(parse_answer("no answer here") is None, "parse missing")

cot, ans = get_cot_and_answer(ds["train"][0])
t.check(isinstance(cot, str) and len(cot) > 10, "cot is string")
t.check(isinstance(ans, float), "answer is float")
t.summary("Data Loading")

print(f"\nExample question: {ds['train'][0]['question'][:100]}...")
print(f"Answer: {ans}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Train: 7473, Test: 1319

[Data Loading] 6/6 tests passed — ALL PASSED
  PASS: parse simple
  PASS: parse comma
  PASS: parse negative
  PASS: parse missing
  PASS: cot is string
  PASS: answer is float

Example question: Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How m...
Answer: 72.0


In [5]:
# Cell 5: Model loading — SDPA attention + explicit device placement
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # left-pad for causal LM

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.float16,
    attn_implementation="sdpa",  # use PyTorch SDPA for faster attention
).to(device)
model.eval()

# Register forward hook on the final layernorm (after all transformer layers)
# This gives us the same output as hidden_states[-1] from output_hidden_states=True
_last_hidden = {}
def _hook_fn(module, input, output):
    _last_hidden["val"] = output

_hook_handle = model.model.norm.register_forward_hook(_hook_fn)

# Test
t = TestResults()
test_input = tokenizer("Hello", return_tensors="pt").to(device)
with torch.no_grad():
    _ = model(**test_input)
last_hidden = _last_hidden["val"]
t.check(last_hidden.shape[-1] == CONFIG["hidden_dim"], f"hidden dim = {last_hidden.shape[-1]}")
t.check(not torch.isnan(last_hidden).any(), "no NaN in hidden states")
t.summary("Model Loading")
print(f"Model loaded: {CONFIG['model_name']} (SDPA + hook-based extraction)")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


[Model Loading] 2/2 tests passed — ALL PASSED
  PASS: hidden dim = 896
  PASS: no NaN in hidden states
Model loaded: Qwen/Qwen2.5-0.5B-Instruct (SDPA + hook-based extraction)


In [6]:
# Cell 6: Hidden state extraction functions (hook-based, no output_hidden_states overhead)

@torch.no_grad()
def extract_hidden_states(texts: List[str], batch_size: int = 32) -> torch.Tensor:
    """Extract last-layer, last-token hidden states via forward hook.
    Returns: (N, hidden_dim) float32 tensor on CPU.
    """
    all_states = []
    n_batches = (len(texts) + batch_size - 1) // batch_size
    n_truncated = 0
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tokenizer(
            batch, return_tensors="pt", padding=True, truncation=True,
            max_length=CONFIG["max_length"],
        ).to(device)
        # Count truncated examples
        for j in range(len(batch)):
            if inputs["attention_mask"][j].sum() == CONFIG["max_length"]:
                n_truncated += 1
        # Forward pass — hook captures last layer output
        _ = model(**inputs)
        hidden = _last_hidden["val"]  # (B, seq_len, hidden_dim)
        # With left-padding, last token is always the last position
        last_token_states = hidden[:, -1, :]  # (B, hidden_dim)
        all_states.append(last_token_states.float().cpu())
        if (i // batch_size) % 25 == 0:
            print(f"    batch {i // batch_size + 1}/{n_batches}")
    if n_truncated > 0:
        print(f"    WARNING: {n_truncated}/{len(texts)} examples truncated at max_length={CONFIG['max_length']}")
    return torch.cat(all_states, dim=0)

# Tests
t = TestResults()
test_texts = ["What is 2+2?", "The answer to 2+2 is 4 because addition."]
test_states = extract_hidden_states(test_texts, batch_size=2)
t.check(test_states.shape == (2, CONFIG["hidden_dim"]), f"shape: {test_states.shape}")
t.check(not torch.isnan(test_states).any(), "no NaN")
t.check(not torch.isinf(test_states).any(), "no Inf")
cos_sim = F.cosine_similarity(test_states[0:1], test_states[1:2]).item()
t.check(cos_sim < 0.99, f"states are distinct (cos_sim={cos_sim:.4f})")
t.summary("Hidden State Extraction")

    batch 1/1

[Hidden State Extraction] 4/4 tests passed — ALL PASSED
  PASS: shape: torch.Size([2, 896])
  PASS: no NaN
  PASS: no Inf
  PASS: states are distinct (cos_sim=0.8922)


In [7]:
# Cell 7: Phase 1 — Run extraction on GSM8K (with caching) + free LLM after

cache_dir = Path(CONFIG["cache_dir"])
cache_dir.mkdir(exist_ok=True)

def run_extraction(split, n_examples):
    """Extract condition (problem) and target (problem+CoT) hidden states."""
    cache_file = cache_dir / f"{split}_{n_examples}.pt"
    if cache_file.exists():
        print(f"Loading cached {split} data from {cache_file}")
        return torch.load(cache_file, weights_only=False)

    data = ds[split].select(range(min(n_examples, len(ds[split]))))
    problems = []
    problems_with_cot = []
    answers = []
    skipped = 0

    for ex in data:
        q = ex["question"]
        cot, ans = get_cot_and_answer(ex)
        if ans is None:
            skipped += 1
            continue
        problems.append(q)
        problems_with_cot.append(f"{q}\n{cot}")
        answers.append(ans)

    if skipped > 0:
        print(f"  Skipped {skipped} examples with unparseable answers")
    print(f"Extracting {len(problems)} {split} examples...")

    print("  Extracting condition (problem-only) hidden states...")
    h_cond = extract_hidden_states(problems, CONFIG["batch_size_extract"])

    print("  Extracting target (problem+CoT) hidden states (smaller batches for long seqs)...")
    h_target = extract_hidden_states(problems_with_cot, CONFIG["batch_size_extract_long"])

    answers_tensor = torch.tensor(answers, dtype=torch.float32)

    result = {"h_cond": h_cond, "h_target": h_target, "answers": answers_tensor}
    torch.save(result, cache_file)
    print(f"  Saved to {cache_file}")
    return result

train_data = run_extraction("train", CONFIG["n_train"])
test_data = run_extraction("test", CONFIG["n_test"])

# Free LLM from GPU — it's not needed for Phase 2/3
_hook_handle.remove()
del model, tokenizer, _last_hidden
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()
    print(f"\nLLM freed. GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
else:
    print("\nLLM freed from memory.")

# z-score normalization (fit on train)
h_mean = train_data["h_target"].mean(dim=0)
h_std = train_data["h_target"].std(dim=0).clamp(min=1e-6)
a_mean = train_data["answers"].mean()
a_std = train_data["answers"].std().clamp(min=1e-6)

def normalize_h(h):
    return (h - h_mean) / h_std

def normalize_a(a):
    return (a - a_mean) / a_std

def denormalize_a(a):
    return a * a_std + a_mean

# Tests
t = TestResults()
for name, d in [("train", train_data), ("test", test_data)]:
    t.check(d["h_cond"].shape[1] == CONFIG["hidden_dim"], f"{name} cond dim")
    t.check(d["h_target"].shape[1] == CONFIG["hidden_dim"], f"{name} target dim")
    t.check(not torch.isnan(d["h_cond"]).any(), f"{name} cond no NaN")
    t.check(not torch.isnan(d["h_target"]).any(), f"{name} target no NaN")
    cos = F.cosine_similarity(d["h_cond"], d["h_target"]).mean().item()
    t.check(cos < 0.99, f"{name} cond != target (mean cos={cos:.4f})")

t.check(train_data["h_cond"].shape[0] >= CONFIG["n_train"] * 0.9, "enough train examples")
t.check(test_data["h_cond"].shape[0] >= CONFIG["n_test"] * 0.9, "enough test examples")
t.summary("Phase 1: Extraction")

print(f"\nTrain: {train_data['h_cond'].shape[0]} examples")
print(f"Test: {test_data['h_cond'].shape[0]} examples")
print(f"Hidden dim: {CONFIG['hidden_dim']}")

Extracting 2000 train examples...
  Extracting condition (problem-only) hidden states...
    batch 1/63
    batch 26/63
    batch 51/63
  Extracting target (problem+CoT) hidden states (smaller batches for long seqs)...
    batch 1/500
    batch 26/500
    batch 51/500
    batch 76/500
    batch 101/500
    batch 126/500
    batch 151/500
    batch 176/500
    batch 201/500
    batch 226/500
    batch 251/500
    batch 276/500
    batch 301/500
    batch 326/500
    batch 351/500
    batch 376/500
    batch 401/500
    batch 426/500
    batch 451/500
    batch 476/500
  Saved to flowthought_cache/train_2000.pt
Extracting 500 test examples...
  Extracting condition (problem-only) hidden states...
    batch 1/16
  Extracting target (problem+CoT) hidden states (smaller batches for long seqs)...
    batch 1/125
    batch 26/125
    batch 51/125
    batch 76/125
    batch 101/125
  Saved to flowthought_cache/test_500.pt

LLM freed. GPU allocated: 0.01 GB

[Phase 1: Extraction] 12/12 tests pa

In [ ]:
# Cell 8: VelocityMLP definition (with DDPM-style fixed time embedding)
import math

class TimestepEmbedding(nn.Module):
    """DDPM-style sinusoidal timestep embedding with fixed log-linear frequencies.
    
    Maps t from (B, 1) to (B, embed_dim) using fixed geometric frequencies
    (matching the standard from Ho et al. 2020 / Vaswani et al. 2017),
    followed by a 2-layer MLP with SiLU activation.
    """
    def __init__(self, embed_dim=64):
        super().__init__()
        half = embed_dim // 2
        # Fixed log-linear frequencies: exp(-log(10000) * i / (half-1))
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half, dtype=torch.float32) / max(half - 1, 1))
        self.register_buffer("freqs", freqs)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, embed_dim),
            nn.SiLU(),
            nn.Linear(embed_dim, embed_dim),
        )

    def forward(self, t):
        # t: (B, 1)
        freq_t = t * self.freqs.unsqueeze(0)  # (B, half)
        emb = torch.cat([torch.sin(freq_t), torch.cos(freq_t)], dim=-1)  # (B, embed_dim)
        return self.mlp(emb)


class VelocityMLP(nn.Module):
    """MLP velocity field for CFM: v_theta(h_t, t, c) -> dh/dt"""

    def __init__(self, hidden_dim, mlp_hidden, n_layers, time_embed_dim=64):
        super().__init__()
        self.time_embed = TimestepEmbedding(time_embed_dim)
        input_dim = hidden_dim * 2 + time_embed_dim
        layers = []
        layers.append(nn.Linear(input_dim, mlp_hidden))
        layers.append(nn.SiLU())
        for _ in range(n_layers - 1):
            layers.append(nn.Linear(mlp_hidden, mlp_hidden))
            layers.append(nn.LayerNorm(mlp_hidden))
            layers.append(nn.SiLU())
        layers.append(nn.Linear(mlp_hidden, hidden_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, h_t, t, c):
        t_emb = self.time_embed(t)
        x = torch.cat([h_t, t_emb, c], dim=-1)
        return self.net(x)

# Tests — only correctness checks, no quality thresholds
t = TestResults()
test_mlp = VelocityMLP(
    CONFIG["hidden_dim"], CONFIG["fm_hidden"], CONFIG["fm_layers"],
    time_embed_dim=CONFIG["time_embed_dim"],
).to(device)
B = 4
h_t = torch.randn(B, CONFIG["hidden_dim"], device=device)
t_in = torch.rand(B, 1, device=device)
c = torch.randn(B, CONFIG["hidden_dim"], device=device)
out = test_mlp(h_t, t_in, c)
t.check(out.shape == (B, CONFIG["hidden_dim"]), f"output shape: {out.shape}")
t.check(not torch.isnan(out).any(), "no NaN")
t.check(not torch.isinf(out).any(), "no Inf")
n_params = sum(p.numel() for p in test_mlp.parameters())
t.check(n_params > 0, f"has {n_params:,} parameters")
t.summary("VelocityMLP")
del test_mlp

In [ ]:
# Cell 9: CFM loss function (with logit-normal time sampling)

def cfm_loss(velocity_net, h_target, h_cond):
    """
    Conditional Flow Matching loss with logit-normal time sampling.
    - Source: h_0 ~ N(0, I)
    - Target: h_1 = normalized CoT hidden states
    - Condition: c = normalized problem hidden states
    - Linear interpolation: h_t = (1-t)*h_0 + t*h_1
    - True velocity: u_t = h_1 - h_0
    - Loss: MSE(v_theta(h_t, t, c), u_t)

    Uses logit-normal time sampling (Meta Flow Matching Guide, 2024) which
    biases training toward intermediate timesteps where velocity is hardest.
    """
    B = h_target.shape[0]
    h_0 = torch.randn_like(h_target)  # source noise

    # Logit-normal time sampling: t = sigmoid(N(0, 1)), biases toward t=0.5
    t = torch.sigmoid(torch.randn(B, 1, device=h_target.device))

    # Linear interpolation
    h_t = (1 - t) * h_0 + t * h_target

    # True velocity (linear path)
    u_t = h_target - h_0

    # Predicted velocity
    v_pred = velocity_net(h_t, t, h_cond)

    return F.mse_loss(v_pred, u_t)

# Test: loss decreases over a few steps
t = TestResults()
test_net = VelocityMLP(CONFIG["hidden_dim"], 256, 2).to(device)
test_opt = torch.optim.Adam(test_net.parameters(), lr=1e-3)

h_tgt_small = normalize_h(train_data["h_target"][:32]).to(device)
h_cnd_small = normalize_h(train_data["h_cond"][:32]).to(device)

losses = []
for step in range(20):
    test_opt.zero_grad()
    loss = cfm_loss(test_net, h_tgt_small, h_cnd_small)
    loss.backward()
    test_opt.step()
    losses.append(loss.item())

t.check(losses[-1] < losses[0], f"loss decreased: {losses[0]:.4f} -> {losses[-1]:.4f}")
t.check(not np.isnan(losses[-1]), "loss not NaN")
t.summary("CFM Loss")
del test_net, test_opt

In [ ]:
# Cell 10: Phase 2 — Train FM (AMP + torch.compile + EMA + checkpointing)
import copy

velocity_net = VelocityMLP(
    CONFIG["hidden_dim"], CONFIG["fm_hidden"], CONFIG["fm_layers"],
    time_embed_dim=CONFIG["time_embed_dim"],
).to(device)

# torch.compile for kernel fusion — test with a dummy forward pass to catch lazy failures
velocity_net_compiled = velocity_net
if device.type == "cuda":
    try:
        _candidate = torch.compile(velocity_net)
        _test_out = _candidate(
            torch.randn(2, CONFIG["hidden_dim"], device=device),
            torch.rand(2, 1, device=device),
            torch.randn(2, CONFIG["hidden_dim"], device=device),
        )
        del _test_out
        velocity_net_compiled = _candidate
        print("torch.compile: enabled")
    except Exception as e:
        print(f"torch.compile: disabled ({e})")

# EMA copy for stable inference
ema_net = copy.deepcopy(velocity_net)
ema_decay = 0.999

optimizer = torch.optim.AdamW(velocity_net.parameters(), lr=CONFIG["fm_lr"], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["fm_epochs"])
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

# Prepare normalized data — all on GPU (only ~14MB total)
h_target_train = normalize_h(train_data["h_target"]).to(device)
h_cond_train = normalize_h(train_data["h_cond"]).to(device)

n_train = h_target_train.shape[0]
loss_history = []
checkpoint_dir = cache_dir / "fm_checkpoints"
checkpoint_dir.mkdir(exist_ok=True)

# Check for existing checkpoint to resume from
start_epoch = 0
latest_ckpt = checkpoint_dir / "latest.pt"
if latest_ckpt.exists():
    ckpt = torch.load(latest_ckpt, weights_only=False)
    try:
        velocity_net.load_state_dict(ckpt["model"])
        ema_net.load_state_dict(ckpt["ema"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        if "scaler" in ckpt:
            scaler.load_state_dict(ckpt["scaler"])
        loss_history = ckpt["loss_history"]
        start_epoch = ckpt["epoch"] + 1
        print(f"Resumed from checkpoint at epoch {start_epoch}")
    except RuntimeError as e:
        print(f"Checkpoint incompatible (architecture changed?), training from scratch: {e}")
        latest_ckpt.unlink()
        loss_history = []
        start_epoch = 0
        # Reinitialize optimizer/scheduler/scaler to clean state
        optimizer = torch.optim.AdamW(velocity_net.parameters(), lr=CONFIG["fm_lr"], weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["fm_epochs"])
        scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)

print(f"Training FM: epochs {start_epoch}-{CONFIG['fm_epochs']}, {n_train} examples, batch_size={CONFIG['fm_batch_size']}")
print(f"VelocityMLP params: {sum(p.numel() for p in velocity_net.parameters()):,}")
if device.type == "cuda":
    print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

for epoch in range(start_epoch, CONFIG["fm_epochs"]):
    perm = torch.randperm(n_train, device=device)
    epoch_losses = []

    for i in range(0, n_train, CONFIG["fm_batch_size"]):
        idx = perm[i:i + CONFIG["fm_batch_size"]]
        h_tgt = h_target_train[idx]
        h_cnd = h_cond_train[idx]

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type, enabled=USE_AMP):
            loss = cfm_loss(velocity_net_compiled, h_tgt, h_cnd)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(velocity_net.parameters(), CONFIG["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        epoch_losses.append(loss.item())

    # EMA update
    with torch.no_grad():
        for p_ema, p_model in zip(ema_net.parameters(), velocity_net.parameters()):
            p_ema.mul_(ema_decay).add_(p_model, alpha=1 - ema_decay)

    scheduler.step()
    avg_loss = np.mean(epoch_losses)
    loss_history.append(avg_loss)

    if (epoch + 1) % 50 == 0 or epoch == start_epoch:
        print(f"  Epoch {epoch+1:3d}/{CONFIG['fm_epochs']} | Loss: {avg_loss:.6f} | LR: {scheduler.get_last_lr()[0]:.2e}")

    # Checkpoint
    if (epoch + 1) % CONFIG["fm_checkpoint_every"] == 0:
        torch.save({
            "epoch": epoch,
            "model": velocity_net.state_dict(),
            "ema": ema_net.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "scaler": scaler.state_dict(),
            "loss_history": loss_history,
        }, latest_ckpt)

# Plot
plt.figure(figsize=(8, 4))
plt.semilogy(loss_history)
plt.xlabel("Epoch")
plt.ylabel("Loss (log)")
plt.title("FM Training Loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Tests — hard-check loss decreases at all, soft-check magnitude
t = TestResults()
pct_decrease = (loss_history[0] - loss_history[-1]) / loss_history[0] * 100
t.check(loss_history[-1] < loss_history[0], f"loss decreased: {loss_history[0]:.4f} -> {loss_history[-1]:.4f} ({pct_decrease:.0f}%)")
t.warn(pct_decrease > 25, f"loss decreased by >25% (got {pct_decrease:.0f}%) — increase epochs/data for stronger signal")
t.summary("Phase 2: FM Training")

In [11]:
# Cell 11: ODE integration (RK4 — same quality as Euler-50 in ~20 steps)

@torch.no_grad()
def rk4_integrate(velocity_net, h_cond, n_steps=20, device=None):
    """
    RK4-integrate the learned velocity field from t=0 to t=1.
    h_cond: (B, hidden_dim) — normalized condition vectors
    Returns: (B, hidden_dim) — generated hidden states at t=1
    """
    if device is None:
        device = h_cond.device
    B, D = h_cond.shape
    dt = 1.0 / n_steps
    h_t = torch.randn(B, D, device=device)

    for step in range(n_steps):
        t_val = step * dt
        t1 = torch.full((B, 1), t_val, device=device)
        t2 = torch.full((B, 1), t_val + dt / 2, device=device)
        t3 = torch.full((B, 1), t_val + dt, device=device)

        k1 = velocity_net(h_t, t1, h_cond)
        k2 = velocity_net(h_t + k1 * dt / 2, t2, h_cond)
        k3 = velocity_net(h_t + k2 * dt / 2, t2, h_cond)
        k4 = velocity_net(h_t + k3 * dt, t3, h_cond)

        h_t = h_t + (k1 + 2 * k2 + 2 * k3 + k4) * dt / 6

    return h_t

# Tests — use EMA net for inference (more stable)
t = TestResults()
test_cond = normalize_h(test_data["h_cond"][:16]).to(device)
generated = rk4_integrate(ema_net, test_cond, CONFIG["ode_steps"])
t.check(generated.shape == (16, CONFIG["hidden_dim"]), f"shape: {generated.shape}")
t.check(not torch.isnan(generated).any(), "no NaN")
t.check(not torch.isinf(generated).any(), "no Inf")
pairwise_cos = F.cosine_similarity(generated[:-1], generated[1:]).mean().item()
t.check(pairwise_cos < 0.99, f"not collapsed (avg pairwise cos={pairwise_cos:.4f})")
t.summary("RK4 ODE Integration")


[RK4 ODE Integration] 4/4 tests passed — ALL PASSED
  PASS: shape: torch.Size([16, 896])
  PASS: no NaN
  PASS: no Inf
  PASS: not collapsed (avg pairwise cos=0.0072)


In [12]:
# Cell 12: Answer probe — MLP trained to predict answer from hidden states

class AnswerProbe(nn.Module):
    def __init__(self, hidden_dim, probe_hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, probe_hidden),
            nn.ReLU(),
            nn.Linear(probe_hidden, probe_hidden),
            nn.ReLU(),
            nn.Linear(probe_hidden, 1),
        )

    def forward(self, h):
        return self.net(h).squeeze(-1)

# Train probe on real CoT hidden states
probe = AnswerProbe(CONFIG["hidden_dim"], CONFIG["probe_hidden"]).to(device)
probe_opt = torch.optim.Adam(probe.parameters(), lr=CONFIG["probe_lr"])

h_target_norm = normalize_h(train_data["h_target"]).to(device)
a_norm = normalize_a(train_data["answers"]).to(device)

probe_losses = []
for epoch in range(CONFIG["probe_epochs"]):
    perm = torch.randperm(h_target_norm.shape[0], device=device)
    epoch_loss = []
    for i in range(0, h_target_norm.shape[0], 128):
        idx = perm[i:i+128]
        pred = probe(h_target_norm[idx])
        loss = F.mse_loss(pred, a_norm[idx])
        probe_opt.zero_grad()
        loss.backward()
        probe_opt.step()
        epoch_loss.append(loss.item())
    probe_losses.append(np.mean(epoch_loss))

probe.eval()

# Test on train set
with torch.no_grad():
    train_pred = denormalize_a(probe(h_target_norm)).cpu()
    train_true = train_data["answers"]
    train_corr = np.corrcoef(train_pred.numpy(), train_true.numpy())[0, 1]

t = TestResults()
t.check(probe_losses[-1] < probe_losses[0], f"probe loss decreased: {probe_losses[0]:.4f} -> {probe_losses[-1]:.4f}")
t.check(not np.isnan(train_corr), f"train correlation is finite: {train_corr:.4f}")
t.warn(train_corr > 0.1, f"train correlation > 0.1 (got {train_corr:.4f}) — may need more data")
t.summary("Answer Probe")
print(f"Probe train correlation: {train_corr:.4f}")


[Answer Probe] 3/3 tests passed — ALL PASSED
  PASS: probe loss decreased: 1.0073 -> 0.0000
  PASS: train correlation is finite: 1.0000
  PASS: train correlation > 0.1 (got 1.0000) — may need more data
Probe train correlation: 1.0000


In [13]:
# Cell 13: Full evaluation — 4 baselines (using EMA net + RK4)

def evaluate_method(name, h_states, true_answers):
    """Evaluate hidden states via the answer probe."""
    with torch.no_grad():
        h_norm = normalize_h(h_states).to(device)
        pred_norm = probe(h_norm).cpu()
        pred = denormalize_a(pred_norm)
    true = true_answers.numpy()
    pred_np = pred.numpy()

    corr = np.corrcoef(pred_np, true)[0, 1] if len(true) > 1 else 0.0
    exact = np.mean(np.abs(np.round(pred_np) - true) < 0.5)
    mse = np.mean((pred_np - true) ** 2)

    return {"name": name, "correlation": corr, "exact_match": exact, "mse": mse}

# Prepare test data
test_answers = test_data["answers"]
n_test = test_data["h_cond"].shape[0]

# Method 1: No-CoT (condition/problem hidden states only)
r_nocot = evaluate_method("No-CoT", test_data["h_cond"], test_answers)

# Method 2: Real CoT (ground truth target hidden states)
r_real = evaluate_method("Real CoT", test_data["h_target"], test_answers)

# Method 3: Random (Gaussian noise)
r_random = evaluate_method("Random", torch.randn(n_test, CONFIG["hidden_dim"]) * h_std + h_mean, test_answers)

# Method 4: FlowThought (FM-generated hidden states via EMA + RK4)
h_cond_test_norm = normalize_h(test_data["h_cond"]).to(device)
h_flow = rk4_integrate(ema_net, h_cond_test_norm, CONFIG["ode_steps"])
h_flow_denorm = (h_flow.cpu() * h_std + h_mean)
r_flow = evaluate_method("FlowThought", h_flow_denorm, test_answers)

results = [r_nocot, r_real, r_random, r_flow]

# Diversity analysis
def diversity_score(h):
    """Mean pairwise cosine distance (1 - cos_sim) on a subsample."""
    h_sub = h[:min(200, len(h))]
    h_norm_vec = F.normalize(h_sub.float(), dim=-1)
    sim_matrix = h_norm_vec @ h_norm_vec.T
    mask = ~torch.eye(len(h_sub), dtype=torch.bool)
    return (1 - sim_matrix[mask].mean()).item()

div_real = diversity_score(test_data["h_target"])
div_flow = diversity_score(h_flow_denorm)
div_random = diversity_score(torch.randn(n_test, CONFIG["hidden_dim"]))

# Distribution match: mean/std distance between flow and real
real_mean = normalize_h(test_data["h_target"]).mean(dim=0)
flow_mean = h_flow.cpu().mean(dim=0)
mean_dist = (real_mean - flow_mean).norm().item()

real_std = normalize_h(test_data["h_target"]).std(dim=0)
flow_std = h_flow.cpu().std(dim=0)
std_dist = (real_std - flow_std).norm().item()

print("\n" + "="*70)
print("EVALUATION RESULTS")
print("="*70)
print(f"{'Method':<15} {'Correlation':>12} {'Exact Match':>12} {'MSE':>12}")
print("-"*51)
for r in results:
    print(f"{r['name']:<15} {r['correlation']:>12.4f} {r['exact_match']:>12.4f} {r['mse']:>12.1f}")

print(f"\nDiversity (cosine distance):")
print(f"  Real CoT: {div_real:.4f}")
print(f"  FlowThought: {div_flow:.4f}")
print(f"  Random: {div_random:.4f}")

print(f"\nDistribution match (FlowThought vs Real CoT):")
print(f"  Mean distance: {mean_dist:.4f}")
print(f"  Std distance: {std_dist:.4f}")

# Hard checks: no crashes, valid outputs
# Soft checks: quality metrics (may be weak with small data)
t = TestResults()
t.check(not np.isnan(r_flow["correlation"]), "FlowThought correlation not NaN")
t.check(not np.isnan(r_real["correlation"]), "Real CoT correlation not NaN")
t.check(div_flow > 0.001, f"FlowThought not fully collapsed (diversity={div_flow:.4f})")
t.warn(r_flow["correlation"] > r_random["correlation"],
       f"FlowThought > random ({r_flow['correlation']:.4f} vs {r_random['correlation']:.4f}) — scale up data if WARN")
t.warn(r_real["correlation"] > r_random["correlation"],
       f"Real CoT > random ({r_real['correlation']:.4f} vs {r_random['correlation']:.4f}) — probe may need more data")
t.summary("Full Evaluation")


EVALUATION RESULTS
Method           Correlation  Exact Match          MSE
---------------------------------------------------
No-CoT               -0.0033       0.0000 21276341043200.0
Real CoT              0.1627       0.0000 3705309233152.0
Random               -0.0172       0.0000 38400442761216.0
FlowThought          -0.0315       0.0000 48205689520128.0

Diversity (cosine distance):
  Real CoT: 0.5303
  FlowThought: 0.5524
  Random: 0.9997

Distribution match (FlowThought vs Real CoT):
  Mean distance: 4.3063
  Std distance: 1.8522

[Full Evaluation] 4/5 tests passed — PASSED with 1 warning(s)
  PASS: FlowThought correlation not NaN
  PASS: Real CoT correlation not NaN
  PASS: FlowThought not fully collapsed (diversity=0.5524)
  WARN: FlowThought > random (-0.0315 vs -0.0172) — scale up data if WARN
  PASS: Real CoT > random (0.1627 vs -0.0172) — probe may need more data


In [ ]:
# Cell: Phase 3b — Reload LLM for Coconut-style hidden state injection
# Strategy: representation engineering — add scaled FM-generated hidden state
# to the residual stream at an intermediate layer during the PREFILL pass only.
# The hook deactivates after the first forward pass so decode steps are unaffected.

from transformers import AutoTokenizer, AutoModelForCausalLM

print("Reloading LLM for injection experiments...")
tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    torch_dtype=torch.float16,
    attn_implementation="sdpa",
).to(device)
model.eval()

num_layers = len(model.model.layers)
inject_layer = num_layers // 2
print(f"Model has {num_layers} layers, injecting at layer {inject_layer}")

# Injection state — mutable dict accessed by hook
# "fired" tracks whether the prefill pass has happened; after that, hook is inactive
injection_state = {"active": False, "vector": None, "alpha": 0.1, "fired": False}

def injection_hook(module, input, output):
    """Add scaled FM state to residual stream at last token position.
    Only fires ONCE per generation (the prefill pass), then deactivates.
    This prevents compounding injection across decode steps.
    """
    if not injection_state["active"] or injection_state["fired"]:
        return output
    hidden = output[0]  # (B, seq_len, hidden_dim) — tuple from decoder layer
    vec = injection_state["vector"].to(hidden.device, hidden.dtype)
    hidden = hidden.clone()
    hidden[:, -1, :] = hidden[:, -1, :] + injection_state["alpha"] * vec
    injection_state["fired"] = True  # deactivate after prefill
    return (hidden,) + output[1:]

hook_handle = model.model.layers[inject_layer].register_forward_hook(injection_hook)

# Tests
t = TestResults()
t.check(num_layers > 0, f"model has {num_layers} layers")
t.check(inject_layer == num_layers // 2, f"inject_layer = {inject_layer}")
test_input = tokenizer("Hello", return_tensors="pt").to(device)
with torch.no_grad():
    _ = model(**test_input)
t.check(True, "forward pass works with hook registered")
t.summary("LLM Reload for Injection")
if device.type == "cuda":
    print(f"GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Cell: Generation helpers for injection evaluation

@torch.no_grad()
def generate_answer(question, inject_vector=None, alpha=0.1, max_new_tokens=50):
    """Generate an answer with optional hidden-state injection.
    
    Args:
        question: problem text
        inject_vector: (hidden_dim,) tensor — DENORMALIZED hidden state in LLM's native space
        alpha: scaling factor for injection
        max_new_tokens: generation budget
    Returns:
        generated text (answer portion only)
    """
    prompt = f"Question: {question}\nAnswer: The answer is"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=CONFIG["max_length"]).to(device)
    
    if inject_vector is not None:
        injection_state["active"] = True
        injection_state["fired"] = False  # reset for new generation
        injection_state["vector"] = inject_vector.unsqueeze(0)  # (1, hidden_dim)
        injection_state["alpha"] = alpha
    else:
        injection_state["active"] = False
        injection_state["fired"] = False
    
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=1.0,
    )
    
    injection_state["active"] = False
    generated = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return generated

def parse_generated_answer(text):
    """Extract first number from generated text."""
    match = re.search(r'[-+]?\d[\d,]*\.?\d*', text.strip())
    if match:
        try:
            return float(match.group().replace(",", ""))
        except Exception:
            return None
    return None

# Sanity test: generation works, injection changes output
t = TestResults()
sample_q = ds["test"][0]["question"]

gen_baseline = generate_answer(sample_q, inject_vector=None)
t.check(isinstance(gen_baseline, str) and len(gen_baseline) > 0, f"baseline generates text: '{gen_baseline[:60]}...'")

# Use DENORMALIZED flow states for injection (h_flow_denorm from cell 13)
gen_injected = generate_answer(sample_q, inject_vector=h_flow_denorm[0], alpha=0.3)
t.check(isinstance(gen_injected, str) and len(gen_injected) > 0, f"injected generates text: '{gen_injected[:60]}...'")

# Verify hook only fired once (prefill-only)
t.check(injection_state["fired"], "hook fired during generation")

parsed = parse_generated_answer("The answer is 42.")
t.check(parsed == 42.0, f"parse_generated_answer works: {parsed}")
parsed2 = parse_generated_answer("1,234 apples")
t.check(parsed2 == 1234.0, f"parse comma number: {parsed2}")

t.summary("Generation Helpers")

In [ ]:
# Cell: Phase 3b — Run injection evaluation across alpha values
# Compare: baseline (no injection) vs FlowThought vs random injection
# Uses DENORMALIZED FM states (h_flow_denorm) and prefill-only injection

n_eval = min(100, test_data["h_cond"].shape[0])
test_questions = ds["test"].select(range(n_eval))
alphas_to_test = [0.01, 0.05, 0.1, 0.3]

print(f"Evaluating {n_eval} test examples across {len(alphas_to_test)} alpha values...")
print(f"Injection layer: {inject_layer}/{num_layers} (middle layer)")
print(f"FM states: denormalized to LLM's native hidden-state space")
print(f"Injection: prefill-only (hook deactivates after first forward pass)")
print()

# --- Baseline (no injection) ---
print("Running baseline (no injection)...")
baseline_results = []
for i in range(n_eval):
    q = test_questions[i]["question"]
    true_ans = parse_answer(test_questions[i]["answer"])
    if true_ans is None:
        continue
    gen = generate_answer(q, inject_vector=None)
    pred = parse_generated_answer(gen)
    baseline_results.append({"true": true_ans, "pred": pred, "text": gen})
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{n_eval}")

baseline_correct = sum(1 for r in baseline_results if r["pred"] is not None and abs(r["pred"] - r["true"]) < 0.5)
baseline_total = len(baseline_results)
print(f"Baseline: {baseline_correct}/{baseline_total} = {baseline_correct/baseline_total:.4f}\n")

# --- FlowThought injection at multiple alphas (using DENORMALIZED states) ---
alpha_results = {}
for alpha in alphas_to_test:
    print(f"Running FlowThought injection (alpha={alpha})...")
    ft_results = []
    for i in range(n_eval):
        q = test_questions[i]["question"]
        true_ans = parse_answer(test_questions[i]["answer"])
        if true_ans is None:
            continue
        flow_vec = h_flow_denorm[i]  # denormalized FM state in LLM's native space
        gen = generate_answer(q, inject_vector=flow_vec, alpha=alpha)
        pred = parse_generated_answer(gen)
        ft_results.append({"true": true_ans, "pred": pred, "text": gen})
        if (i + 1) % 25 == 0:
            print(f"  {i+1}/{n_eval}")
    
    correct = sum(1 for r in ft_results if r["pred"] is not None and abs(r["pred"] - r["true"]) < 0.5)
    total = len(ft_results)
    alpha_results[alpha] = {"results": ft_results, "correct": correct, "total": total}
    print(f"  FlowThought (alpha={alpha}): {correct}/{total} = {correct/total:.4f}")

# --- Random injection (control, alpha=0.1) with matching scale ---
# Use random vectors with same mean/std as real CoT hidden states
print(f"\nRunning random injection (alpha=0.1, control)...")
random_results = []
for i in range(n_eval):
    q = test_questions[i]["question"]
    true_ans = parse_answer(test_questions[i]["answer"])
    if true_ans is None:
        continue
    rand_vec = torch.randn(CONFIG["hidden_dim"]) * h_std + h_mean  # match real distribution scale
    gen = generate_answer(q, inject_vector=rand_vec, alpha=0.1)
    pred = parse_generated_answer(gen)
    random_results.append({"true": true_ans, "pred": pred, "text": gen})
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{n_eval}")

random_correct = sum(1 for r in random_results if r["pred"] is not None and abs(r["pred"] - r["true"]) < 0.5)
random_total = len(random_results)
print(f"Random: {random_correct}/{random_total} = {random_correct/random_total:.4f}")

# === Results Table ===
print("\n" + "=" * 70)
print("PHASE 3b: HIDDEN STATE INJECTION RESULTS")
print("=" * 70)
print(f"{'Method':<30} {'Correct':>8} {'Total':>6} {'Accuracy':>10}")
print("-" * 56)
print(f"{'Baseline (no injection)':<30} {baseline_correct:>8} {baseline_total:>6} {baseline_correct/baseline_total:>10.4f}")
for alpha in alphas_to_test:
    ar = alpha_results[alpha]
    label = f"FlowThought (alpha={alpha})"
    print(f"{label:<30} {ar['correct']:>8} {ar['total']:>6} {ar['correct']/ar['total']:>10.4f}")
print(f"{'Random (alpha=0.1)':<30} {random_correct:>8} {random_total:>6} {random_correct/random_total:>10.4f}")

# === Sample outputs ===
print("\n--- Sample outputs (first 3 examples) ---")
for i in range(min(3, len(baseline_results))):
    print(f"\nExample {i+1} | True answer: {baseline_results[i]['true']}")
    print(f"  Baseline:    {baseline_results[i]['text'][:80]}")
    best_alpha = alphas_to_test[1]  # show alpha=0.05
    print(f"  FT(a={best_alpha}):  {alpha_results[best_alpha]['results'][i]['text'][:80]}")
    print(f"  Random:      {random_results[i]['text'][:80]}")

# === Tests ===
t = TestResults()
t.check(baseline_total > 0, f"evaluated {baseline_total} examples")
t.check(not any(np.isnan(r["true"]) for r in baseline_results), "no NaN in true answers")

# Check injection actually changes some outputs (not all identical to baseline)
n_changed = sum(1 for i in range(min(len(baseline_results), len(alpha_results[0.3]["results"])))
                if baseline_results[i]["text"] != alpha_results[0.3]["results"][i]["text"])
t.check(n_changed > 0, f"injection changes {n_changed}/{len(baseline_results)} outputs at alpha=0.3")

# Check no NaN in FM vectors used
t.check(not torch.isnan(h_flow_denorm[:n_eval]).any().item(), "no NaN in FM-generated states")

# Soft check: FlowThought at some alpha >= baseline or >= random
best_ft_acc = max(ar["correct"] / ar["total"] for ar in alpha_results.values())
t.warn(best_ft_acc >= random_correct / random_total,
       f"best FlowThought ({best_ft_acc:.4f}) >= random ({random_correct/random_total:.4f})")

t.summary("Phase 3b: Injection Evaluation")

# Cleanup
hook_handle.remove()
del model, tokenizer
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()
    print(f"\nLLM freed. GPU allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
# Cell 14: Results summary — charts + JSON export

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

names = [r["name"] for r in results]
colors = ["#888", "#2ecc71", "#e74c3c", "#3498db"]

# Correlation
axes[0].bar(names, [r["correlation"] for r in results], color=colors)
axes[0].set_title("Correlation with True Answer")
axes[0].set_ylabel("Pearson r")
axes[0].tick_params(axis='x', rotation=15)

# Exact match
axes[1].bar(names, [r["exact_match"] for r in results], color=colors)
axes[1].set_title("Exact Match Rate")
axes[1].set_ylabel("Accuracy")
axes[1].tick_params(axis='x', rotation=15)

# Diversity
div_names = ["Real CoT", "FlowThought", "Random"]
div_vals = [div_real, div_flow, div_random]
axes[2].bar(div_names, div_vals, color=["#2ecc71", "#3498db", "#e74c3c"])
axes[2].set_title("Diversity (Cosine Distance)")
axes[2].set_ylabel("Mean Pairwise Distance")
axes[2].tick_params(axis='x', rotation=15)

plt.suptitle("FlowThought PoC Results", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Save results
summary = {
    "config": CONFIG,
    "results": results,
    "diversity": {"real": div_real, "flow": div_flow, "random": div_random},
    "distribution_match": {"mean_dist": mean_dist, "std_dist": std_dist},
    "fm_final_loss": loss_history[-1],
    "probe_train_corr": train_corr,
}

with open("flowthought_results.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print("\nResults saved to flowthought_results.json")
print("\n" + "="*70)
print("INTERPRETATION")
print("="*70)
print(f"""\n
Phase 3a (probe-based evaluation) results:
- FlowThought correlation: {r_flow['correlation']:.4f} (vs random: {r_random['correlation']:.4f})
- Real CoT correlation: {r_real['correlation']:.4f} (upper bound)
- FlowThought did NOT outperform random on the probe metric — the probe-based
  evaluation is inconclusive for measuring whether FM learns useful structure.
- Diversity {div_flow:.4f} shows FM generates varied states (not mode-collapsed)
- Distribution match (mean dist: {mean_dist:.4f}) indicates gap between FM and real CoT distribution

The probe evaluation has a fundamental limitation: it measures linear readout of
answer information, which is a weak proxy for whether hidden states carry useful
reasoning signal. Phase 3b (Coconut-style injection) is the real test — it
measures whether FM-generated states actually improve downstream generation
accuracy when injected into the LLM residual stream.

Next steps for stronger signal:
1. Larger velocity networks
2. More training data
3. RLVR fine-tuning (Flow-GRPO)
4. Better injection strategies (multi-layer, learned alpha)
""")